# 🏥 Karaciğer Fibrozunun BT Görüntülerinden Non-İnvaziv Evrelendirilmesi

## Professional Demonstration Notebook

**Öğrenci**: Bülent Tuğrul (22290673)  
**Kurum**: Ankara Üniversitesi - Bilgisayar Mühendisliği

---

### 🎯 Projenin CAN ALICI NOKTALARI:

1. **2D FFT Frekans Domain Analizi** - Maskeleme etkisini kırar
2. **NASH Tespiti** - HU-based steatosis detection
3. **Hibrit Feature Fusion** - Spatial + Frequency birleştirme
4. **XGBoost + SHAP** - Açıklanabilir AI modeli

---

## 1. Setup ve Import

In [ ]:
# Core imports
import numpy as np
import matplotlib.pyplot as plt
import sys
from pathlib import Path

# Add src to path
sys.path.append('../src')

# Import our modules
from data_processing.dicom_loader import DICOMLoader
from feature_extraction.frequency_domain.fft_2d import FFT2DAnalyzer
from feature_extraction.spatial_domain.nash_detection import NASHDetector
from feature_extraction.spatial_domain.radiomics_features import RadiomicsExtractor
from models.deep_learning.unet_segmentation import TraditionalSegmentor

print("✅ Modules imported successfully!")

## 2. DICOM Veri Yükleme

In [ ]:
# Initialize DICOM loader
dataset_path = "../data/raw/TCIA-DATASET-DICOM"
loader = DICOMLoader(dataset_path)

print(f"Found {len(loader.dicom_files)} DICOM files")

if loader.dicom_files:
    # Load first file
    first_file = loader.dicom_files[0]
    image_hu, metadata = loader.load_dicom(first_file)
    
    print(f"\nImage Information:")
    print(f"  Patient ID: {metadata.patient_id}")
    print(f"  Shape: {image_hu.shape}")
    print(f" HU Range: [{image_hu.min():.1f}, {image_hu.max():.1f}]")
    
    # Visualize
    windowed = loader.apply_window(image_hu, 'liver')
    
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    axes[0].imshow(image_hu, cmap='gray')
    axes[0].set_title('Original CT (HU values)')
    axes[0].axis('off')
    
    axes[1].imshow(windowed, cmap='gray')
    axes[1].set_title('Liver Window Applied')
    axes[1].axis('off')
    
    plt.tight_layout()
    plt.show()

## 3. Karaciğer Segmentasyonu

In [ ]:
# Segment liver
segmentor = TraditionalSegmentor()
liver_mask = segmentor.segment_liver_traditional(image_hu)
spleen_mask = segmentor.segment_spleen_traditional(image_hu, liver_mask)

print(f"Liver area: {np.sum(liver_mask)} pixels")
print(f"Spleen area: {np.sum(spleen_mask)} pixels")

# Visualize
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

axes[0].imshow(windowed, cmap='gray')
axes[0].set_title('Original')
axes[0].axis('off')

axes[1].imshow(windowed, cmap='gray')
axes[1].imshow(liver_mask, alpha=0.3, cmap='Reds')
axes[1].set_title('Liver Segmentation')
axes[1].axis('off')

axes[2].imshow(windowed, cmap='gray')
axes[2].imshow(spleen_mask, alpha=0.3, cmap='Blues')
axes[2].set_title('Spleen Segmentation')
axes[2].axis('off')

plt.tight_layout()
plt.show()

## 4. ⭐ 2D FFT Frekans Domain Analizi (CAN ALICI NOKTA #1)

In [ ]:
# Extract liver ROI
liver_roi = image_hu * liver_mask
liver_roi_norm = ((liver_roi - liver_roi.min()) / (liver_roi.max() - liver_roi.min() + 1e-8) * 255).astype(np.uint8)

# Initialize FFT analyzer
fft_analyzer = FFT2DAnalyzer(window_function='hamming')

# Extract FFT features
fft_features = fft_analyzer.extract_all_features(liver_roi_norm)

print("="*60)
print("FFT FEATURES (Frequency Domain)")
print("="*60)
print(f"Low/High Frequency Ratio: {fft_features.low_high_ratio:.4f}")
print(f"  → >2.0 suggests fibrosis")
print(f"\nNASH Frequency Signature: {fft_features.steatosis_frequency_signature:.4f}")
print(f"  → >0.5 suggests NASH")
print(f"\nAnisotropy Index: {fft_features.anisotropy_index:.4f}")
print(f"  → Higher value indicates directional fibrosis")
print(f"\nSpectral Entropy: {fft_features.spectral_entropy:.4f}")
print(f"  → Higher entropy = more heterogeneous tissue")

# Visualize FFT
fft_analyzer.visualize_fft(liver_roi_norm, show_plot=True)

## 5. ⭐ NASH Detection (CAN ALICI NOKTA #2)

In [ ]:
# Initialize NASH detector
nash_detector = NASHDetector()

# Extract NASH features
nash_features = nash_detector.extract_all_features(image_hu, liver_mask, spleen_mask)

print("="*60)
print("NASH DETECTION RESULTS (Spatial Domain)")
print("="*60)
print(f"\n🎯 NASH Probability Score: {nash_features.nash_probability_score:.2%}")
print(f"   Confidence: {nash_features.nash_confidence.upper()}")

print(f"\n📊 Key Indicators:")
print(f"   Steatosis %: {nash_features.steatosis_percentage:.1f}%")
if nash_features.steatosis_percentage > 30:
    print(f"     → SEVERE steatosis")
elif nash_features.steatosis_percentage > 10:
    print(f"     → MODERATE steatosis")
else:
    print(f"     → MILD/NO steatosis")

print(f"\n   Liver/Spleen Ratio: {nash_features.liver_spleen_ratio:.2f}")
if nash_features.liver_spleen_ratio < 0.8:
    print(f"     → ABNORMAL - Significant steatosis")
elif nash_features.liver_spleen_ratio < 1.0:
    print(f"     → BORDERLINE")
else:
    print(f"     → NORMAL")

print(f"\n   Mean HU: {nash_features.mean_hu:.1f}")
print(f"   Hepatomegaly Score: {nash_features.hepatomegaly_score:.2f}")

## 6. PyRadiomics Texture Features

In [ ]:
# Initialize radiomics extractor
radiomics = RadiomicsExtractor(bin_width=25, normalize=True)

# Extract features
radiomics_features = radiomics.extract_features(image_hu, liver_mask)

print(f"Extracted {len(radiomics_features)} radiomics features")

# Group features
grouped = radiomics.extract_feature_groups(image_hu, liver_mask)

print("\nFeature Groups:")
for group, feats in grouped.items():
    if feats:
        print(f"  {group.upper()}: {len(feats)} features")

## 7. ⭐ Feature Fusion (CAN ALICI NOKTA #3)

In [ ]:
# Combine all features
all_features = {}

# FFT features
fft_dict = {f'fft_{k}': v for k, v in fft_features.__dict__.items()}
all_features.update(fft_dict)

# NASH features
nash_dict = {f'nash_{k}': v for k, v in nash_features.__dict__.items() 
             if isinstance(v, (int, float))}
all_features.update(nash_dict)

# Radiomics features (sample)
radiomics_sample = {f'rad_{k}': v for k, v in list(radiomics_features.items())[:20]}
all_features.update(radiomics_sample)

print("="*60)
print("HYBRID FEATURE FUSION")
print("="*60)
print(f"\nTotal Features: {len(all_features)}")
print(f"  - FFT (Frequency): {len(fft_dict)}")
print(f"  - NASH (Spatial): {len(nash_dict)}")
print(f"  - Radiomics (Sample): {len(radiomics_sample)}")

print("\nFeature Vector ready for XGBoost classification! 🚀")

## 8. Clinical Interpretation Summary

In [ ]:
print("\n" + "="*70)
print("CLINICAL INTERPRETATION SUMMARY")
print("="*70)

print(f"\n👤 Patient: {metadata.patient_id}")
print(f"📅 Study Date: {metadata.study_date}")
print(f"🔬 Series: {metadata.series_description}")

print(f"\n📊 NASH Assessment:")
print(f"   Probability: {nash_features.nash_probability_score:.1%}")
print(f"   Confidence: {nash_features.nash_confidence.upper()}")

print(f"\n🔍 Key Findings:")
print(f"   • Steatosis: {nash_features.steatosis_percentage:.1f}%")
print(f"   • L/S Ratio: {nash_features.liver_spleen_ratio:.2f}")
print(f"   • FFT Low/High: {fft_features.low_high_ratio:.2f}")
print(f"   • Spectral Entropy: {fft_features.spectral_entropy:.2f}")

print(f"\n💡 Interpretation:")
if nash_features.nash_probability_score > 0.7:
    print(f"   ⚠️ HIGH probability of NASH")
elif nash_features.nash_probability_score > 0.4:
    print(f"   ⚡ MODERATE probability of NASH")
else:
    print(f"   ✅ LOW probability of NASH")

if fft_features.low_high_ratio > 2.0:
    print(f"   ⚠️ FFT analysis suggests possible fibrosis")

print(f"\n📝 Recommendation:")
print(f"   → Feed these features to XGBoost model for fibrosis staging")
print(f"   → Use SHAP for explainability")
print(f"   → Compare with biopsy ground truth (if available)")

print("\n" + "="*70)

## 🎉 Sonuç

Bu notebook, **Karaciğer Fibrozunun BT Görüntülerinden Non-İnvaziv Evrelendirilmesi** projesinin temel özelliklerini göstermektedir:

### ✅ Başarıyla Gösterilen Özellikler:

1. **2D FFT Frekans Domain Analizi**
   - Spektral entropi
   - Low/High frequency ratio
   - NASH frequency signature
   - Directional anisotropy

2. **NASH Detection**
   - HU-based steatosis quantification
   - Liver/Spleen ratio
   - Morphometric features
   - NASH probability scoring

3. **PyRadiomics Integration**
   - 100+ texture features
   - GLCM, GLRLM, GLSZM
   - Shape and intensity features

4. **Hybrid Feature Fusion**
   - Frekans + Uzamsal birleştirme
   - 100+ özellikli feature vector
   - XGBoost için hazır

###  Sonraki Adımlar:

1. Tüm TCIA hastaları için feature extraction
2. Ground truth labels ile eşleştirme
3. XGBoost model eğitimi
4. SHAP analysis
5. K-fold cross validation
6. Final thesis report

---

**📧 Bülent Tuğrul - 22290673**  
**🏛️ Ankara Üniversitesi - Bilgisayar Mühendisliği**
